In [26]:
from datasets import load_dataset

ds1 = load_dataset('parquet', data_files='deepscaler/train.parquet', split='train')
ds2 = load_dataset('parquet', data_files='simplelr_math_35/train.parquet', split='train')

print(len(ds1))
print(len(ds2))


40315
8523


In [27]:
p = 0.2
n1 = int(len(ds1) * p)
n2 = int(len(ds2) * p)
print(n1, n2)

ds1_sft = ds1.shuffle(seed=42).select(range(n1))
ds1_remaining = ds1.shuffle(seed=42).select(range(n1, len(ds1)))

ds2_sft = ds2.shuffle(seed=42).select(range(n2))
ds2_remaining = ds2.shuffle(seed=42).select(range(n2, len(ds2)))


8063 1704


In [28]:
# 检查 ds1_sft 的结构
print("ds1_sft[0]:")
print(ds1_sft[0])
print("\nds1_sft extra_info 结构:")
print(ds1_sft[0]['extra_info'])
print("\nds1_sft extra_info 的字段:", list(ds1_sft[0]['extra_info'].keys()) if isinstance(ds1_sft[0]['extra_info'], dict) else "Not a dict")

ds1_sft[0]:
{'data_source': 'deepscaler', 'prompt': [{'content': 'Automobile license plates for a state consist of four letters followed by a dash and two single digits. How many different license plate combinations are possible if exactly one letter is repeated exactly once, but digits cannot be repeated? [asy]\nsize(150);\ndraw((0,0)--(0,5)--(10,5)--(10,0)--cycle);\nlabel("\\Huge{CHIC - 03}",(1,3)--(9,3),S);\nlabel("\\small\\emph{State of Excellence}",(1,1)--(9,1),S);\ndraw((0.5,3.5)--(0.5,4.5)--(2,4.5)--(2,3.5)--cycle);\nlabel("\\footnotesize 5-03",(1.25,4));\ndraw((9.5,3.5)--(9.5,4.5)--(8,4.5)--(8,3.5)--cycle);\nlabel("\\footnotesize FX",(8.75,4));\n[/asy]', 'role': 'user'}], 'ability': 'math', 'reward_model': {'ground_truth': '8,\\!424,\\!000', 'style': 'rule'}, 'extra_info': {'index': 35379, 'split': 'train'}}

ds1_sft extra_info 结构:
{'index': 35379, 'split': 'train'}

ds1_sft extra_info 的字段: ['index', 'split']


In [29]:
# 检查 ds2_sft 的原始结构
print("ds2_sft[0] (原始):")
print(ds2_sft[0])
print("\nds2_sft extra_info 结构:")
print(ds2_sft[0]['extra_info'])
print("\nds2_sft extra_info 的字段:", list(ds2_sft[0]['extra_info'].keys()) if isinstance(ds2_sft[0]['extra_info'], dict) else "Not a dict")

ds2_sft[0] (原始):
{'input': '<|im_start|>system\nPlease reason step by step, and put your final answer within \\boxed{}.<|im_end|>\n<|im_start|>user\n$x$ is a real number with the property that $x+\\tfrac1x = 3$. Let $S_m = x^m + \\tfrac{1}{x^m}$. Determine the value of $S_7$.<|im_end|>\n<|im_start|>assistant', 'gt_answer': '843', 'subject': 'Intermediate Algebra', 'ground_truth_answer': '843', 'target': '843', 'data_source': 'simplelr_math_35', 'prompt': [{'content': '$x$ is a real number with the property that $x+\\tfrac1x = 3$. Let $S_m = x^m + \\tfrac{1}{x^m}$. Determine the value of $S_7$.', 'role': 'user'}], 'ability': 'math', 'reward_model': {'ground_truth': '843', 'style': 'rule'}, 'extra_info': {'answer': '843', 'index': 6693, 'level': 5, 'question': '$x$ is a real number with the property that $x+\\tfrac1x = 3$. Let $S_m = x^m + \\tfrac{1}{x^m}$. Determine the value of $S_7$.', 'split': 'train'}}

ds2_sft extra_info 结构:
{'answer': '843', 'index': 6693, 'level': 5, 'question': 

In [30]:
# 先检查 ds1_sft 的 extra_info 结构，确保完全匹配
print("ds1_sft[0]['extra_info']:", ds1_sft[0]['extra_info'])
print("ds1_sft extra_info 字段:", list(ds1_sft[0]['extra_info'].keys()))

# 方法：先移除 extra_info，然后重新添加，确保结构完全匹配
from datasets import Features, Value

# 定义新的 extra_info 结构（只包含 index 和 split）
new_extra_info_features = {
    'index': Value('int64'),
    'split': Value('string')
}

def reorg(example):
    # 只保留需要的字段，并创建新的 extra_info
    return {
        'data_source': example['data_source'],
        'prompt': example['prompt'],
        'ability': example['ability'],
        'reward_model': example['reward_model'],
        # 创建新的 extra_info，只包含 index 和 split
        'new_extra_info': {
            'index': example['extra_info']['index'],
            'split': 'train'
        }
    }

# 先重组数据，将 extra_info 重命名为 new_extra_info
ds2_sft = ds2_sft.map(reorg, remove_columns=ds2_sft.column_names)

# 移除旧的 extra_info（如果还存在），并将 new_extra_info 重命名为 extra_info
def rename_extra_info(example):
    return {
        'extra_info': example['new_extra_info']
    }

ds2_sft = ds2_sft.map(rename_extra_info, remove_columns=['new_extra_info'])

# 现在明确指定 extra_info 的 features
from datasets import Features
target_extra_info_features = Features({
    'index': Value('int64'),
    'split': Value('string')
})

# 使用 cast_column 来明确转换 extra_info 的结构
ds2_sft = ds2_sft.cast_column('extra_info', target_extra_info_features)

print("\n重组后的 ds2_sft[0]:")
print(ds2_sft[0])
print("\n重组后的 ds2_sft extra_info:", ds2_sft[0]['extra_info'])

# 验证两个数据集的 features 是否一致
print("\n检查 features 是否一致:")
print("ds1_sft extra_info features:", ds1_sft.features['extra_info'])
print("ds2_sft extra_info features:", ds2_sft.features['extra_info'])
print("\n是否一致:", ds1_sft.features == ds2_sft.features)


ds1_sft[0]['extra_info']: {'index': 35379, 'split': 'train'}
ds1_sft extra_info 字段: ['index', 'split']


Map: 100%|██████████| 1704/1704 [00:00<00:00, 16463.90 examples/s]


重组后的 ds2_sft[0]:
{'data_source': 'simplelr_math_35', 'prompt': [{'content': '$x$ is a real number with the property that $x+\\tfrac1x = 3$. Let $S_m = x^m + \\tfrac{1}{x^m}$. Determine the value of $S_7$.', 'role': 'user'}], 'ability': 'math', 'reward_model': {'ground_truth': '843', 'style': 'rule'}, 'extra_info': {'index': 6693, 'split': 'train'}}

重组后的 ds2_sft extra_info: {'index': 6693, 'split': 'train'}

检查 features 是否一致:
ds1_sft extra_info features: {'index': Value(dtype='int64', id=None), 'split': Value(dtype='string', id=None)}
ds2_sft extra_info features: {'index': Value(dtype='int64', id=None), 'split': Value(dtype='string', id=None)}

是否一致: True


In [31]:
# 合并前最后检查 features 是否一致
print("合并前检查:")
print("ds1_sft features:", ds1_sft.features)
print("ds2_sft features:", ds2_sft.features)
print("\n是否一致:", ds1_sft.features == ds2_sft.features)

# 如果仍然不一致，使用 cast 统一整个 features
if ds1_sft.features != ds2_sft.features:
    print("\nFeatures 仍然不一致，使用 cast 统一整个 features...")
    ds2_sft = ds2_sft.cast(ds1_sft.features)
    print("转换后的 ds2_sft features:", ds2_sft.features)

# 现在可以合并了
from datasets import concatenate_datasets
ds_sft = concatenate_datasets([ds1_sft, ds2_sft])
print(f"\n✅ 合并成功！合并后的数据集大小: {len(ds_sft)}")
print(f"合并后的 features: {ds_sft.features}")

合并前检查:
ds1_sft features: {'data_source': Value(dtype='string', id=None), 'prompt': [{'content': Value(dtype='string', id=None), 'role': Value(dtype='string', id=None)}], 'ability': Value(dtype='string', id=None), 'reward_model': {'ground_truth': Value(dtype='string', id=None), 'style': Value(dtype='string', id=None)}, 'extra_info': {'index': Value(dtype='int64', id=None), 'split': Value(dtype='string', id=None)}}
ds2_sft features: {'data_source': Value(dtype='string', id=None), 'prompt': [{'content': Value(dtype='string', id=None), 'role': Value(dtype='string', id=None)}], 'ability': Value(dtype='string', id=None), 'reward_model': {'ground_truth': Value(dtype='string', id=None), 'style': Value(dtype='string', id=None)}, 'extra_info': {'index': Value(dtype='int64', id=None), 'split': Value(dtype='string', id=None)}}

是否一致: True

✅ 合并成功！合并后的数据集大小: 9767
合并后的 features: {'data_source': Value(dtype='string', id=None), 'prompt': [{'content': Value(dtype='string', id=None), 'role': Value(dtype

In [32]:
len(ds_sft)

9767

In [33]:

ds_sft[0]

{'data_source': 'deepscaler',
 'prompt': [{'content': 'Automobile license plates for a state consist of four letters followed by a dash and two single digits. How many different license plate combinations are possible if exactly one letter is repeated exactly once, but digits cannot be repeated? [asy]\nsize(150);\ndraw((0,0)--(0,5)--(10,5)--(10,0)--cycle);\nlabel("\\Huge{CHIC - 03}",(1,3)--(9,3),S);\nlabel("\\small\\emph{State of Excellence}",(1,1)--(9,1),S);\ndraw((0.5,3.5)--(0.5,4.5)--(2,4.5)--(2,3.5)--cycle);\nlabel("\\footnotesize 5-03",(1.25,4));\ndraw((9.5,3.5)--(9.5,4.5)--(8,4.5)--(8,3.5)--cycle);\nlabel("\\footnotesize FX",(8.75,4));\n[/asy]',
   'role': 'user'}],
 'ability': 'math',
 'reward_model': {'ground_truth': '8,\\!424,\\!000', 'style': 'rule'},
 'extra_info': {'index': 35379, 'split': 'train'}}

In [34]:
ds_sft.to_parquet('sft.parquet')

Creating parquet from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 35.80ba/s]


3097601

In [35]:
ds1_remaining.to_parquet('deepscaler/train_remaining.parquet')
ds2_remaining.to_parquet('simplelr_math_35/train_remaining.parquet')

Creating parquet from Arrow format: 100%|██████████| 7/7 [00:00<00:00, 19.48ba/s]


6605456